# Load Libraries

In [1]:
import pandas as pd
import numpy as np

from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

In [2]:
customers = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/olist_customers_dataset.csv')
location = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/olist_sellers_dataset.csv')
category_name_translation = pd.read_csv('/Users/sahil_jangid/codes/Main_projects/data_analytics/resume/retail-consumer-intelligence-platform/data/raw/product_category_name_translation.csv')

In [8]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "location": location,
    "category_name_translation": category_name_translation
}

# Data Type Assessment

In [3]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],

    "order_items": [
        "shipping_limit_date"
    ],

    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

datetime_columns

{'orders': ['order_purchase_timestamp',
  'order_approved_at',
  'order_delivered_carrier_date',
  'order_delivered_customer_date',
  'order_estimated_delivery_date'],
 'order_items': ['shipping_limit_date'],
 'reviews': ['review_creation_date', 'review_answer_timestamp']}

In [4]:
datasets = {
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews
}

for dataset_name, columns in datetime_columns.items():

    df = datasets[dataset_name]

    for col in columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

In [5]:
for dataset_name, columns in datetime_columns.items():

    print(f"\n{dataset_name.upper()}")

    display(
        datasets[dataset_name][columns].dtypes
    )


ORDERS


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


ORDER_ITEMS


shipping_limit_date    datetime64[us]
dtype: object


REVIEWS


review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

# Missing Value Analysis

In [9]:
missing_report = []

for name, df in datasets.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    for column, count in missing.items():

        missing_report.append({
            "Dataset": name,
            "Column": column,
            "Missing Values": count,
            "Missing (%)": round((count / len(df)) * 100, 2)
        })

missing_report = pd.DataFrame(missing_report)

missing_report.sort_values(
    by="Missing (%)",
    ascending=False,
    ignore_index=True
)

,Dataset,Column,Missing Values,Missing (%)
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


# Cleaning Decisions

In [10]:
cleaning_decisions = pd.DataFrame({

    "Dataset": [
        "Orders",
        "Orders",
        "Orders",
        "Reviews",
        "Reviews",
        "Products",
        "Products"
    ],

    "Column": [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "review_comment_title",
        "review_comment_message",
        "product_category_name",
        "product_dimensions"
    ],

    "Reason": [
        "Cancelled or unavailable orders",
        "Order not shipped",
        "Order not delivered",
        "Optional field",
        "Optional field",
        "Missing metadata",
        "Missing metadata"
    ],

    "Action": [
        "Keep NaN",
        "Keep NaN",
        "Keep NaN",
        "Keep NaN",
        "Keep NaN",
        "Investigate",
        "Investigate"
    ]

})

cleaning_decisions

,Dataset,Column,Reason,Action
0,Orders,order_approved_at,Cancelled or unavailable orders,Keep NaN
1,Orders,order_delivered_carrier_date,Order not shipped,Keep NaN
2,Orders,order_delivered_customer_date,Order not delivered,Keep NaN
3,Reviews,review_comment_title,Optional field,Keep NaN
4,Reviews,review_comment_message,Optional field,Keep NaN
5,Products,product_category_name,Missing metadata,Investigate
6,Products,product_dimensions,Missing metadata,Investigate


In [11]:
orders.loc[
    orders["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [12]:
orders.loc[
    orders["order_approved_at"].isna(),
    "order_status"
].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [13]:
orders.loc[
    orders["order_delivered_carrier_date"].isna(),
    "order_status"
].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64


# Referential Integrity Validation

In [15]:
validation_results = []

relationships = [
    (
        "orders",
        "customer_id",
        orders["customer_id"],
        customers["customer_id"]
    ),

    (
        "order_items",
        "product_id",
        order_items["product_id"],
        products["product_id"]
    ),

    (
        "order_items",
        "seller_id",
        order_items["seller_id"],
        sellers["seller_id"]
    ),

    (
        "payments",
        "order_id",
        payments["order_id"],
        orders["order_id"]
    ),

    (
        "reviews",
        "order_id",
        reviews["order_id"],
        orders["order_id"]
    )
]

for table, column, child, parent in relationships:

    invalid = (~child.isin(parent)).sum()

    validation_results.append({
        "Table": table,
        "Foreign Key": column,
        "Invalid Records": invalid,
        "Status": "Pass" if invalid == 0 else "Fail"
    })

validation_results = pd.DataFrame(validation_results)

validation_results

,Table,Foreign Key,Invalid Records,Status
0,orders,customer_id,0,Pass
1,order_items,product_id,0,Pass
2,order_items,seller_id,0,Pass
3,payments,order_id,0,Pass
4,reviews,order_id,0,Pass


# Business Rule Validation

In [16]:
invalid_delivery = orders[
    orders["order_delivered_customer_date"] <
    orders["order_purchase_timestamp"]
]

print(f"Invalid delivery dates: {len(invalid_delivery)}")

invalid_delivery.head()

Invalid delivery dates: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [17]:
invalid_delivery = orders[
    orders["order_delivered_customer_date"] <
    orders["order_purchase_timestamp"]
]

print(f"Invalid delivery dates: {len(invalid_delivery)}")

invalid_delivery.head()

Invalid delivery dates: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [18]:
invalid_estimate = orders[
    orders["order_estimated_delivery_date"] <
    orders["order_purchase_timestamp"]
]

print(f"Invalid estimated delivery dates: {len(invalid_estimate)}")

invalid_estimate.head()

Invalid estimated delivery dates: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [19]:
products[products["product_weight_g"] < 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [20]:
products[products["product_weight_g"] < 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [21]:
order_items[order_items["freight_value"] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [22]:
reviews[
    ~reviews["review_score"].between(1, 5)
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [23]:
payments[
    payments["payment_installments"] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


### Business Rule Observation

Two payment records contain `payment_installments = 0`.

Since both transactions have valid payment values and represent an extremely small proportion of the dataset, these records will be retained. They are treated as edge cases rather than data quality errors.

In [24]:
products[
    (products["product_length_cm"] <= 0) |
    (products["product_height_cm"] <= 0) |
    (products["product_width_cm"] <= 0)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


# Export Cleaned Datasets

In [26]:
import os

os.makedirs("../data/cleaned", exist_ok=True)

In [27]:
customers.to_csv("../data/cleaned/customers_cleaned.csv", index=False)

orders.to_csv("../data/cleaned/orders_cleaned.csv", index=False)

order_items.to_csv("../data/cleaned/order_items_cleaned.csv", index=False)

payments.to_csv("../data/cleaned/payments_cleaned.csv", index=False)

reviews.to_csv("../data/cleaned/reviews_cleaned.csv", index=False)

products.to_csv("../data/cleaned/products_cleaned.csv", index=False)

sellers.to_csv("../data/cleaned/sellers_cleaned.csv", index=False)

location.to_csv("../data/cleaned/geolocation_cleaned.csv", index=False)

category_name_translation.to_csv(
    "../data/cleaned/category_translation_cleaned.csv",
    index=False
)